# 06 — Power BI Export: Gold Hand-off Validation

| | |
|---|---|
| **Project** | Charge IQ EV Station Utilization Analytics |
| **Team** | Team 17 |
| **Week** | 9 — Power BI dashboard refinement |
| **Notebook** | `notebooks/06_powerbi_export.ipynb` |
| **Role** | Week-8 → Week-9 Gold hand-off validation / export artifact |

**Power BI dashboard pages**
1. EV Charging Station Analytics Dashboard
2. EV Station Performance & Operations
3. EV Charging Maintenance & Reliability

## 1. Objective

Validate the **same approved Gold hand-off from Week 8** before Power BI refresh/use, and state clearly whether it is ready.

This notebook:
- loads/connects to the **existing** approved Gold tables (read-only),
- validates existence, schema, row counts, nulls, duplicates/grain and date coverage,
- computes total sessions, energy and estimated revenue,
- performs the **February 2026 reconciliation** (Gold `89,484` vs Power BI KPI `89K`),
- reports a final readiness verdict.

This notebook does **not** rebuild the ETL pipeline, create or modify Gold tables, or touch raw / Bronze / Silver data.
Connection-dependent results are shown as **NOT EXECUTED** when the Gold environment is not reachable — no values are invented.

## 2. Week-8 Gold Hand-off

- Power BI uses **only** the two approved Gold tables below.
- Power BI must **not** connect to raw, Bronze or Silver data.
- The two Gold tables remain **independent** — no relationship is created between them.
- The **Activity Date slicer** is based on `gold_overall_daily_activity`; overall visuals respond to it.
- Station-level visuals are independent by design and do not respond to that slicer.

**Main dashboard measures/visuals:** Total EV Charging Sessions · Total Energy (kWh) · Estimated Revenue (INR) · Daily EV Charging Sessions · Station-wise Charging Sessions · Sessions by City Band · Activity Date slicer.

## 3. Approved Gold Tables

**`default.gold_overall_daily_activity`** — grain: one row per `activity_date`  
Columns: `activity_date`, `total_session_count`, `total_energy_kwh`, `estimated_revenue_inr`

**`default.gold_station_daily_activity`** — grain: one row per `station_id` + `activity_date`  
Columns: `station_id`, `station_name`, `city_band`, `zone`, `site_type`, `activity_date`, `daily_session_count`, `daily_energy_kwh`, `daily_estimated_revenue_inr`

## 4. Configuration / Connection

The Gold tables are read through the **Spark session of the Week-8 workflow** (the `spark` object provided by the project's environment). 
No credentials are stored in this notebook. Optional settings are read from environment variables:

| Variable | Purpose | Default |
|---|---|---|
| `GOLD_SCHEMA` | Schema holding the approved Gold tables | `default` |
| `EXPORT_GOLD_CSV` | `true` to export validated Gold tables (only if the workflow requires it) | `false` |
| `GOLD_EXPORT_DIR` | Target folder for the optional export | *(empty — set it yourself)* |

If your environment needs extra connection setup, add it in the placeholder block of the next cell using environment variables only.

In [0]:
import os

# ---------------------------------------------------------------------------
# CONFIGURATION  (no credentials / tokens / passwords belong in this notebook)
# ---------------------------------------------------------------------------
GOLD_SCHEMA = os.environ.get("GOLD_SCHEMA", "default")

TABLES = {
    "overall": f"{GOLD_SCHEMA}.gold_overall_daily_activity",
    "station": f"{GOLD_SCHEMA}.gold_station_daily_activity",
}

REQUIRED_COLUMNS = {
    "overall": ["activity_date", "total_session_count", "total_energy_kwh", "estimated_revenue_inr"],
    "station": ["station_id", "station_name", "city_band", "zone", "site_type",
                "activity_date", "daily_session_count", "daily_energy_kwh",
                "daily_estimated_revenue_inr"],
}

GRAIN_KEYS = {
    "overall": ["activity_date"],
    "station": ["station_id", "activity_date"],
}

# Reporting period and known validation values (from the Week-8/Week-9 documentation)
PERIOD_START, PERIOD_END = "2026-01-01", "2026-03-31"
FEB_START, FEB_END = "2026-02-01", "2026-02-28"
FEB_EXPECTED_SESSIONS = 89484
POWERBI_KPI_DISPLAY = "89K"

KNOWN_DAYS = {
    "2026-01-01": {"sessions": 3705, "energy_kwh": 95026.411, "revenue_inr": 1431200.83},
    "2026-03-31": {"sessions": 1990, "energy_kwh": 51071.094, "revenue_inr": 739181.22},
}
ENERGY_TOL, REVENUE_TOL = 0.001, 0.01

# Optional export (OFF by default; Power BI reads the Gold tables directly)
EXPORT_GOLD_CSV = os.environ.get("EXPORT_GOLD_CSV", "false").lower() == "true"
GOLD_EXPORT_DIR = os.environ.get("GOLD_EXPORT_DIR", "")

# ---------------------------------------------------------------------------
# Result registry (reset on every run, so the notebook is safe to rerun)
# ---------------------------------------------------------------------------
CHECK_ORDER = [
    "Gold overall table available",
    "Gold station table available",
    "Required columns present",
    "Row counts",
    "Null validation",
    "Duplicate/grain validation",
    "Date range validation",
    "Total sessions (overall)",
    "Total energy kWh (overall)",
    "Estimated revenue INR (overall)",
    "Known daily values (2026-01-01, 2026-03-31)",
    "Station Gold metrics",
    "February 2026 sessions",
    "Power BI reconciliation",
    "Power BI Gold hand-off ready",
]
RESULTS = {}

def record(check, result, status):
    RESULTS[check] = (str(result), status)
    print(f"[{status}] {check}: {result}")

def fmt_int(v):
    return f"{int(round(float(v))):,}"

def fmt_num(v, d=3):
    return f"{float(v):,.{d}f}"

def d10(v):
    return str(v)[:10]

for name, table in TABLES.items():
    print(f"{name:8s} -> {table}")
print("Export enabled:", EXPORT_GOLD_CSV)

overall  -> default.gold_overall_daily_activity
station  -> default.gold_station_daily_activity
Export enabled: False


## 5. Load Gold Tables

The notebook uses the existing `spark` session when it is available (e.g. the Week-8 environment). 
Tables are only **read**; nothing is written, created or altered.

In [0]:
# Placeholder for any extra environment-specific connection setup.
# Use environment variables only — do not paste credentials here.

spark = globals().get("spark")
CONNECTION_ERROR = None

if spark is None:
    try:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
    except Exception as exc:
        spark = None
        CONNECTION_ERROR = f"{type(exc).__name__}: {exc}"

def run_sql(sql):
    return [row.asDict() for row in spark.sql(sql).collect()]

def show(rows):
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows))
    except Exception:
        for r in rows:
            print(r)

AVAILABLE = {"overall": False, "station": False}
ACCESS_ERRORS = {}

if spark is None:
    print("NOT EXECUTED: no Spark session / Gold environment is reachable from this notebook.")
    print("Reason:", CONNECTION_ERROR)
else:
    for key, table in TABLES.items():
        try:
            run_sql(f"SELECT 1 FROM {table} LIMIT 1")
            AVAILABLE[key] = True
        except Exception as exc:
            ACCESS_ERRORS[key] = f"{type(exc).__name__}: {str(exc)[:160]}"

def status_for(key):
    if spark is None:
        return "Not connected", "NOT EXECUTED"
    if AVAILABLE[key]:
        return f"{TABLES[key]} accessible", "PASS"
    return f"{TABLES[key]} not accessible ({ACCESS_ERRORS.get(key, 'unknown error')})", "FAIL"

record("Gold overall table available", *status_for("overall"))
record("Gold station table available", *status_for("station"))

def ready(*keys):
    return spark is not None and all(AVAILABLE[k] for k in keys)

[PASS] Gold overall table available: default.gold_overall_daily_activity accessible
[PASS] Gold station table available: default.gold_station_daily_activity accessible


## 6. Gold Table Schema Validation

Each approved table must contain all required columns. Extra columns are reported but do not fail the check.

In [0]:
def schema_of(key):
    fields = spark.sql(f"SELECT * FROM {TABLES[key]} LIMIT 0").schema.fields
    return {f.name.lower(): str(getattr(f.dataType, "simpleString", lambda: f.dataType)()) for f in fields}

if not ready("overall", "station"):
    record("Required columns present", "Gold tables not reachable", "NOT EXECUTED")
else:
    missing_any = False
    for key in TABLES:
        cols = schema_of(key)
        missing = [c for c in REQUIRED_COLUMNS[key] if c not in cols]
        extra = [c for c in cols if c not in REQUIRED_COLUMNS[key]]
        print(f"\n{TABLES[key]}")
        show([{"column": c, "type": cols.get(c, "MISSING")} for c in REQUIRED_COLUMNS[key]])
        if extra:
            print("Extra columns (informational):", extra)
        missing_any = missing_any or bool(missing)
        if missing:
            print("MISSING:", missing)
    record("Required columns present",
           "All required columns present" if not missing_any else "Missing columns found",
           "FAIL" if missing_any else "PASS")


default.gold_overall_daily_activity


,column,type
0,activity_date,date
1,total_session_count,bigint
2,total_energy_kwh,double
3,estimated_revenue_inr,double


Extra columns (informational): ['active_station_count', 'active_charger_count', 'average_energy_kwh', 'average_duration_minutes', 'total_occupied_minutes', 'gold_processed_at']

default.gold_station_daily_activity


,column,type
0,station_id,string
1,station_name,string
2,city_band,string
3,zone,string
4,site_type,string
5,activity_date,date
6,daily_session_count,bigint
7,daily_energy_kwh,double
8,daily_estimated_revenue_inr,double


Extra columns (informational): ['daily_completed_session_count', 'average_session_energy_kwh', 'average_duration_minutes', 'gold_processed_at']
[PASS] Required columns present: All required columns present


## 7. Row Count and Grain Validation

Computes one aggregate query per table: row counts, distinct keys, date range and metric totals.
Grain duplicates are checked in the next section.

In [0]:
METRICS = {}

if not ready("overall", "station"):
    record("Row counts", "Gold tables not reachable", "NOT EXECUTED")
else:
    METRICS["overall"] = run_sql(f"""
        SELECT COUNT(*) AS row_count,
               COUNT(DISTINCT activity_date) AS distinct_dates,
               MIN(activity_date) AS min_date,
               MAX(activity_date) AS max_date,
               SUM(total_session_count) AS total_sessions,
               SUM(total_energy_kwh) AS total_energy_kwh,
               SUM(estimated_revenue_inr) AS total_revenue_inr
        FROM {TABLES['overall']}""")[0]

    METRICS["station"] = run_sql(f"""
        SELECT COUNT(*) AS row_count,
               COUNT(DISTINCT station_id) AS distinct_stations,
               COUNT(DISTINCT activity_date) AS distinct_dates,
               MIN(activity_date) AS min_date,
               MAX(activity_date) AS max_date,
               SUM(daily_session_count) AS total_sessions,
               SUM(daily_energy_kwh) AS total_energy_kwh,
               SUM(daily_estimated_revenue_inr) AS total_revenue_inr
        FROM {TABLES['station']}""")[0]

    ov, st = METRICS["overall"], METRICS["station"]
    print(f"Overall rows : {fmt_int(ov['row_count'])} (distinct activity_date: {fmt_int(ov['distinct_dates'])})")
    print(f"Station rows : {fmt_int(st['row_count'])} (distinct stations: {fmt_int(st['distinct_stations'])}, "
          f"distinct dates: {fmt_int(st['distinct_dates'])})")
    ok = ov["row_count"] > 0 and st["row_count"] > 0
    record("Row counts",
           f"overall={fmt_int(ov['row_count'])}, station={fmt_int(st['row_count'])}",
           "PASS" if ok else "FAIL")

Overall rows : 90 (distinct activity_date: 90)
Station rows : 15,660 (distinct stations: 174, distinct dates: 90)
[PASS] Row counts: overall=90, station=15,660


## 8. Null and Duplicate Checks

- **Nulls:** every required column of both approved tables must be non-null.
- **Grain:** `gold_overall_daily_activity` → one row per `activity_date`; `gold_station_daily_activity` → one row per `station_id` + `activity_date`.

In [0]:
if not ready("overall", "station"):
    record("Null validation", "Gold tables not reachable", "NOT EXECUTED")
    record("Duplicate/grain validation", "Gold tables not reachable", "NOT EXECUTED")
else:
    # ---- Null checks
    null_total = 0
    for key in TABLES:
        exprs = ", ".join(f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in REQUIRED_COLUMNS[key])
        row = run_sql(f"SELECT {exprs} FROM {TABLES[key]}")[0]
        counts = {c: int(v or 0) for c, v in row.items()}
        null_total += sum(counts.values())
        print(f"\nNull counts - {TABLES[key]}")
        show([{"column": c, "null_count": n} for c, n in counts.items()])
    record("Null validation",
           "0 nulls in required columns" if null_total == 0 else f"{null_total} null values found",
           "PASS" if null_total == 0 else "FAIL")

    # ---- Duplicate / grain checks
    dup_total = 0
    for key in TABLES:
        keys = ", ".join(GRAIN_KEYS[key])
        dups = int(run_sql(f"""
            SELECT COUNT(*) AS dup_groups FROM (
                SELECT {keys} FROM {TABLES[key]} GROUP BY {keys} HAVING COUNT(*) > 1
            ) d""")[0]["dup_groups"])
        dup_total += dups
        print(f"{TABLES[key]}  grain ({keys}): duplicate groups = {dups}")
    record("Duplicate/grain validation",
           "0 duplicate grain keys" if dup_total == 0 else f"{dup_total} duplicate grain groups",
           "PASS" if dup_total == 0 else "FAIL")


Null counts - default.gold_overall_daily_activity


,column,null_count
0,activity_date,0
1,total_session_count,0
2,total_energy_kwh,0
3,estimated_revenue_inr,0



Null counts - default.gold_station_daily_activity


,column,null_count
0,station_id,0
1,station_name,0
2,city_band,0
3,zone,0
4,site_type,0
5,activity_date,0
6,daily_session_count,0
7,daily_energy_kwh,0
8,daily_estimated_revenue_inr,0


[PASS] Null validation: 0 nulls in required columns
default.gold_overall_daily_activity  grain (activity_date): duplicate groups = 0
default.gold_station_daily_activity  grain (station_id, activity_date): duplicate groups = 0
[PASS] Duplicate/grain validation: 0 duplicate grain keys


## 9. Date Range Validation

The reporting period must cover **2026-01-01 to 2026-03-31**. The overall table drives the Activity Date slicer, so its minimum and maximum dates must match the period exactly; station dates must fall inside it.

In [0]:
if not ready("overall", "station"):
    record("Date range validation", "Gold tables not reachable", "NOT EXECUTED")
else:
    ov, st = METRICS["overall"], METRICS["station"]
    ov_min, ov_max = d10(ov["min_date"]), d10(ov["max_date"])
    st_min, st_max = d10(st["min_date"]), d10(st["max_date"])
    show([
        {"table": TABLES["overall"], "min_date": ov_min, "max_date": ov_max, "distinct_dates": int(ov["distinct_dates"])},
        {"table": TABLES["station"], "min_date": st_min, "max_date": st_max, "distinct_dates": int(st["distinct_dates"])},
    ])
    overall_ok = (ov_min == PERIOD_START and ov_max == PERIOD_END)
    station_ok = (st_min >= PERIOD_START and st_max <= PERIOD_END)
    record("Date range validation",
           f"{ov_min} to {ov_max}",
           "PASS" if (overall_ok and station_ok) else "FAIL")

,table,min_date,max_date,distinct_dates
0,default.gold_overall_daily_activity,2026-01-01,2026-03-31,90
1,default.gold_station_daily_activity,2026-01-01,2026-03-31,90


[PASS] Date range validation: 2026-01-01 to 2026-03-31


## 10. Overall Gold Metrics

Totals for `gold_overall_daily_activity` and the known daily values (2026-01-01 and 2026-03-31).
No causal explanation is offered for differences between dates.

In [0]:
if not ready("overall"):
    for c in ["Total sessions (overall)", "Total energy kWh (overall)",
              "Estimated revenue INR (overall)", "Known daily values (2026-01-01, 2026-03-31)"]:
        record(c, "Gold overall table not reachable", "NOT EXECUTED")
else:
    ov = METRICS["overall"]
    record("Total sessions (overall)", fmt_int(ov["total_sessions"]), "PASS" if ov["total_sessions"] and ov["total_sessions"] > 0 else "FAIL")
    record("Total energy kWh (overall)", fmt_num(ov["total_energy_kwh"]), "PASS" if ov["total_energy_kwh"] and ov["total_energy_kwh"] > 0 else "FAIL")
    record("Estimated revenue INR (overall)", fmt_num(ov["total_revenue_inr"], 2), "PASS" if ov["total_revenue_inr"] and ov["total_revenue_inr"] > 0 else "FAIL")

    rows, all_ok = [], True
    for day, exp in KNOWN_DAYS.items():
        found = run_sql(f"""
            SELECT SUM(total_session_count) AS sessions,
                   SUM(total_energy_kwh) AS energy_kwh,
                   SUM(estimated_revenue_inr) AS revenue_inr
            FROM {TABLES['overall']}
            WHERE CAST(activity_date AS DATE) = DATE '{day}'""")[0]
        if found["sessions"] is None:
            ok = False
            rows.append({"activity_date": day, "gold_sessions": None, "expected_sessions": exp["sessions"], "match": "MISSING"})
        else:
            ok = (int(round(float(found["sessions"]))) == exp["sessions"]
                  and abs(float(found["energy_kwh"]) - exp["energy_kwh"]) <= ENERGY_TOL
                  and abs(float(found["revenue_inr"]) - exp["revenue_inr"]) <= REVENUE_TOL)
            rows.append({
                "activity_date": day,
                "gold_sessions": fmt_int(found["sessions"]), "expected_sessions": f"{exp['sessions']:,}",
                "gold_energy_kwh": fmt_num(found["energy_kwh"]), "expected_energy_kwh": f"{exp['energy_kwh']:,.3f}",
                "gold_revenue_inr": fmt_num(found["revenue_inr"], 2), "expected_revenue_inr": f"{exp['revenue_inr']:,.2f}",
                "match": "YES" if ok else "NO",
            })
        all_ok = all_ok and ok
    show(rows)
    record("Known daily values (2026-01-01, 2026-03-31)",
           "Match documented values" if all_ok else "Mismatch vs documented values",
           "PASS" if all_ok else "FAIL")

[PASS] Total sessions (overall): 286,902
[PASS] Total energy kWh (overall): 7,310,333.403
[PASS] Estimated revenue INR (overall): 109,663,218.59


,activity_date,gold_sessions,expected_sessions,gold_energy_kwh,expected_energy_kwh,gold_revenue_inr,expected_revenue_inr,match
0,2026-01-01,"3,705","3,705","95,026.411","95,026.411","1,431,200.83","1,431,200.83",YES
1,2026-03-31,"1,990","1,990","51,071.094","51,071.094","739,181.22","739,181.22",YES


[PASS] Known daily values (2026-01-01, 2026-03-31): Match documented values


## 11. Station Gold Metrics

Station-level totals, top stations and sessions by city band from `gold_station_daily_activity`.

> The station table is **independent by design**: it is not related to the overall table and does not respond to the Activity Date slicer. 
> Its totals are shown for information and are **not** forced to equal the overall totals.

In [0]:
if not ready("station"):
    record("Station Gold metrics", "Gold station table not reachable", "NOT EXECUTED")
else:
    st = METRICS["station"]
    print(f"Station table totals - sessions: {fmt_int(st['total_sessions'])}, "
          f"energy kWh: {fmt_num(st['total_energy_kwh'])}, "
          f"estimated revenue INR: {fmt_num(st['total_revenue_inr'], 2)}")

    print("\nTop 10 stations by sessions")
    show(run_sql(f"""
        SELECT station_id, station_name, SUM(daily_session_count) AS sessions
        FROM {TABLES['station']}
        GROUP BY station_id, station_name
        ORDER BY sessions DESC
        LIMIT 10"""))

    print("\nSessions by city band")
    show(run_sql(f"""
        SELECT city_band, SUM(daily_session_count) AS sessions
        FROM {TABLES['station']}
        GROUP BY city_band
        ORDER BY sessions DESC"""))

    record("Station Gold metrics",
           f"{fmt_int(st['distinct_stations'])} stations, {fmt_int(st['total_sessions'])} sessions",
           "PASS" if st["total_sessions"] and st["total_sessions"] > 0 else "FAIL")

Station table totals - sessions: 286,902, energy kWh: 7,310,333.403, estimated revenue INR: 109,663,218.59

Top 10 stations by sessions


,station_id,station_name,sessions
0,STN0054,Mumbai Transit District Charge Hub 054,3049
1,STN0120,Kochi Outer Ring Charge Hub 120,2680
2,STN0086,Mumbai Transit District Charge Hub 086,2457
3,STN0063,Ahmedabad South Charge Hub 063,2416
4,STN0003,Pune Highway Corridor Charge Hub 003,2371
5,STN0035,Pune Highway Corridor Charge Hub 035,2360
6,STN0089,Hyderabad Central Charge Hub 089,2352
7,STN0129,Hyderabad Central Charge Hub 129,2293
8,STN0125,Delhi NCR West Charge Hub 125,2274
9,STN0091,Pune Highway Corridor Charge Hub 091,2267



Sessions by city band


,city_band,sessions
0,Hyderabad,39217
1,Pune,38496
2,Bengaluru,37477
3,Mumbai,36234
4,Delhi NCR,36127
5,Ahmedabad,34714
6,Kochi,33026
7,Chennai,31611


[PASS] Station Gold metrics: 174 stations, 286,902 sessions


## 12. February 2026 Reconciliation

Power BI shows the sessions KPI in abbreviated form (**89K**); the underlying Gold value must be **89,484**. 
The check passes when the Gold sum equals `89,484` **and** rounding it to thousands gives the displayed `89K`.

In [0]:
def abbreviate_kpi(value):
    # Mirrors how Power BI abbreviates a KPI card (thousands).
    v = float(value)
    if v >= 1_000_000:
        return f"{round(v / 1_000_000)}M"
    if v >= 1_000:
        return f"{round(v / 1_000)}K"
    return f"{round(v)}"

if not ready("overall"):
    record("February 2026 sessions", "Gold overall table not reachable", "NOT EXECUTED")
    record("Power BI reconciliation", "Gold overall table not reachable", "NOT EXECUTED")
else:
    feb = run_sql(f"""
        SELECT SUM(total_session_count) AS sessions, COUNT(*) AS days
        FROM {TABLES['overall']}
        WHERE CAST(activity_date AS DATE) BETWEEN DATE '{FEB_START}' AND DATE '{FEB_END}'""")[0]
    feb_sessions = int(round(float(feb["sessions"]))) if feb["sessions"] is not None else None

    if feb_sessions is None:
        record("February 2026 sessions", "No February 2026 rows found", "FAIL")
        record("Power BI reconciliation", "No February 2026 rows found", "FAIL")
    else:
        displayed = abbreviate_kpi(feb_sessions)
        sessions_ok = feb_sessions == FEB_EXPECTED_SESSIONS
        display_ok = displayed == POWERBI_KPI_DISPLAY
        show([{
            "period": f"{FEB_START} to {FEB_END}",
            "days_in_gold": int(feb["days"]),
            "gold_sessions": f"{feb_sessions:,}",
            "expected_gold_sessions": f"{FEB_EXPECTED_SESSIONS:,}",
            "power_bi_display_expected": POWERBI_KPI_DISPLAY,
            "abbreviated_gold_value": displayed,
        }])
        print("Power BI shows the KPI rounded/abbreviated; the Gold value is the exact total.")
        record("February 2026 sessions", f"{feb_sessions:,}", "PASS" if sessions_ok else "FAIL")
        record("Power BI reconciliation", f"{POWERBI_KPI_DISPLAY} vs {feb_sessions:,}",
               "PASS" if (sessions_ok and display_ok) else "FAIL")

,period,days_in_gold,gold_sessions,expected_gold_sessions,power_bi_display_expected,abbreviated_gold_value
0,2026-02-01 to 2026-02-28,28,"89,484","89,484",89K,89K


Power BI shows the KPI rounded/abbreviated; the Gold value is the exact total.
[PASS] February 2026 sessions: 89,484
[PASS] Power BI reconciliation: 89K vs 89,484


## 13. Power BI Readiness Check

The hand-off is ready only if **every** data check above is `PASS`. If any check is `NOT EXECUTED`, readiness is reported as `NOT EXECUTED` — never assumed.

The design rules below are properties of the Power BI model/report and cannot be verified from this notebook; confirm them in Power BI Desktop (Model view).

In [0]:
DESIGN_RULES = [
    "Power BI source = only default.gold_overall_daily_activity and default.gold_station_daily_activity",
    "No raw / Bronze / Silver tables in the Power BI model",
    "No relationship between the overall and station Gold tables",
    "Activity Date slicer is based on gold_overall_daily_activity",
    "Overall visuals respond to the Activity Date slicer",
    "Station-level visuals remain independent by design",
]
print("Manual confirmation in Power BI Desktop (not verified by this notebook):")
for rule in DESIGN_RULES:
    print("  [ ]", rule)

data_checks = [c for c in CHECK_ORDER if c != "Power BI Gold hand-off ready"]
statuses = [RESULTS.get(c, ("", "NOT EXECUTED"))[1] for c in data_checks]

if "FAIL" in statuses:
    record("Power BI Gold hand-off ready", "No", "FAIL")
elif "NOT EXECUTED" in statuses:
    record("Power BI Gold hand-off ready", "NOT EXECUTED", "NOT EXECUTED")
else:
    record("Power BI Gold hand-off ready", "Yes", "PASS")

# Optional export - only when explicitly enabled AND validation passed
if EXPORT_GOLD_CSV:
    if RESULTS["Power BI Gold hand-off ready"][1] != "PASS":
        print("Export skipped: Gold hand-off is not validated.")
    elif not GOLD_EXPORT_DIR:
        print("Export skipped: set GOLD_EXPORT_DIR.")
    else:
        os.makedirs(GOLD_EXPORT_DIR, exist_ok=True)
        for key, table in TABLES.items():
            path = os.path.join(GOLD_EXPORT_DIR, table.split(".")[-1] + ".csv")
            spark.table(table).toPandas().to_csv(path, index=False)
            print("Exported (unchanged Gold data):", path)
else:
    print("\nExport disabled - Power BI reads the approved Gold tables directly (no files written).")

Manual confirmation in Power BI Desktop (not verified by this notebook):
  [ ] Power BI source = only default.gold_overall_daily_activity and default.gold_station_daily_activity
  [ ] No raw / Bronze / Silver tables in the Power BI model
  [ ] No relationship between the overall and station Gold tables
  [ ] Activity Date slicer is based on gold_overall_daily_activity
  [ ] Overall visuals respond to the Activity Date slicer
  [ ] Station-level visuals remain independent by design
[PASS] Power BI Gold hand-off ready: Yes

Export disabled - Power BI reads the approved Gold tables directly (no files written).


## 14. Validation Summary

In [0]:
summary = [
    {"Check": c, "Result": RESULTS.get(c, ("Not run", "NOT EXECUTED"))[0],
     "Status": RESULTS.get(c, ("Not run", "NOT EXECUTED"))[1]}
    for c in CHECK_ORDER
]
show(summary)

width = max(len(r["Check"]) for r in summary)
rwidth = max(len(r["Result"]) for r in summary)
print()
print(f"{'Check':<{width}} | {'Result':<{rwidth}} | Status")
print("-" * (width + rwidth + 14))
for r in summary:
    print(f"{r['Check']:<{width}} | {r['Result']:<{rwidth}} | {r['Status']}")

,Check,Result,Status
0,Gold overall table available,default.gold_overall_daily_activity accessible,PASS
1,Gold station table available,default.gold_station_daily_activity accessible,PASS
2,Required columns present,All required columns present,PASS
3,Row counts,"overall=90, station=15,660",PASS
4,Null validation,0 nulls in required columns,PASS
5,Duplicate/grain validation,0 duplicate grain keys,PASS
6,Date range validation,2026-01-01 to 2026-03-31,PASS
7,Total sessions (overall),"286,902",PASS
8,Total energy kWh (overall),"7,310,333.403",PASS
9,Estimated revenue INR (overall),"109,663,218.59",PASS



Check                                       | Result                                         | Status
-------------------------------------------------------------------------------------------------------
Gold overall table available                | default.gold_overall_daily_activity accessible | PASS
Gold station table available                | default.gold_station_daily_activity accessible | PASS
Required columns present                    | All required columns present                   | PASS
Row counts                                  | overall=90, station=15,660                     | PASS
Null validation                             | 0 nulls in required columns                    | PASS
Duplicate/grain validation                  | 0 duplicate grain keys                         | PASS
Date range validation                       | 2026-01-01 to 2026-03-31                       | PASS
Total sessions (overall)                    | 286,902                                        

## 15. Conclusion

- The notebook validates the **unchanged Week-8 approved Gold hand-off** (`gold_overall_daily_activity`, `gold_station_daily_activity`) before Power BI refresh/use.
- A **PASS** on *Power BI Gold hand-off ready* means: both tables are accessible, schemas and key fields are complete, grain is unique, the reporting period spans 2026-01-01 to 2026-03-31, known daily values match, and February 2026 reconciles (Gold `89,484` sessions = Power BI `89K`, a display rounding only).
- **NOT EXECUTED** means the Gold environment was not reachable from this run; rerun in the Week-8 environment. No values are assumed.
- **FAIL** items must be resolved in the Gold layer by the Week-8 process, not by changing definitions here.
- Power BI continues to use only the two independent Gold tables; the Activity Date slicer drives the overall visuals, and station-level visuals remain independent by design.